# 03 - New Dataset: BloodMNIST (MedMNIST)

Mục tiêu:
- Đánh giá mô hình trên dataset mới ngoài paper gốc.
- Dữ liệu y sinh dạng MNIST-like phù hợp hướng two-step.
- Kiểm tra khả năng tự điều chỉnh từ K0=12 về gần số lớp thật (8 lớp của BloodMNIST).


In [ ]:
import sys
import subprocess

try:
    import medmnist
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'medmnist'])

import pandas as pd
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from medmnist import INFO

sys.path.append('..')
from src.utils import set_seed, get_device, extract_features, train_clustering

set_seed(42)
device = get_device()
print(f'[INFO] device = {device}')


In [ ]:
# Two-step: feature extraction cho BloodMNIST
data_flag = 'bloodmnist'
info = INFO[data_flag]
DataClass = getattr(medmnist, info['python_class'])

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225)),
])

test_ds = DataClass(split='test', transform=transform, download=True)
subset_size = min(3000, len(test_ds))
subset = torch.utils.data.Subset(test_ds, range(subset_size))
loader = DataLoader(subset, batch_size=128, shuffle=False)

resnet = torchvision.models.resnet18(weights=torchvision.models.ResNet18_Weights.DEFAULT)
backbone = nn.Sequential(*list(resnet.children())[:-1]).to(device)

X, y = extract_features(loader, backbone, device)
print('[INFO] BloodMNIST features:', X.shape, '| labels:', y.shape)
print('[INFO] Number of true classes:', len(info['label']))


In [ ]:
# Chạy PnP dynamic K
k_star, acc, nmi, ari = train_clustering(
    features=X, labels=y, k_max=12, device=device,
    epochs=30, lr=1e-3, lambda_param=2.0,
    enable_split=True, enable_merge=True
)

# Baseline fixed-K để đối chiếu
k_star_fix, acc_fix, nmi_fix, ari_fix = train_clustering(
    features=X, labels=y, k_max=12, device=device,
    epochs=30, lr=1e-3, lambda_param=2.0,
    enable_split=False, enable_merge=False
)

results = pd.DataFrame([
    {'Setting': 'PnP dynamic', 'K0': 12, 'K*': int(k_star), 'ACC(%)': acc*100, 'NMI(%)': nmi*100, 'ARI(%)': ari*100},
    {'Setting': 'Fixed-K baseline', 'K0': 12, 'K*': int(k_star_fix), 'ACC(%)': acc_fix*100, 'NMI(%)': nmi_fix*100, 'ARI(%)': ari_fix*100},
])
results


In [ ]:
true_k = len(INFO['bloodmnist']['label'])
summary = pd.DataFrame([
    {'Metric': '|K* - K_true| (PnP dynamic)', 'Value': abs(int(k_star) - true_k)},
    {'Metric': '|K* - K_true| (Fixed-K baseline)', 'Value': abs(int(k_star_fix) - true_k)},
    {'Metric': 'K_true', 'Value': true_k},
])
summary


## Nhận xét cho báo cáo

- BloodMNIST phù hợp định hướng two-step vì là ảnh MNIST-like nhưng mang ngữ cảnh y sinh (tế bào máu).
- Cần phân tích cả chất lượng phân cụm (ACC/NMI/ARI) và khả năng ước lượng K (độ lệch so với K_true=8).
- Nếu kết quả chưa cao, nêu rõ nguyên nhân: backbone không fine-tune theo miền y sinh, ảnh nhỏ 28x28, và cài đặt from-scratch chưa tune sâu.
